In [1]:
"""
NGO Report Scraper
==================
Searches public NGO/ESG repositories for reports mentioning
Tesla, Volkswagen, Disney, or Netflix, then downloads them.

Sources:
  1. WikiRate        — open REST API, no auth needed
  2. Open Sanctions  — public API for corporate misconduct data
  3. GoodJobs First  — subsidy/accountability tracker (HTML scrape)
  4. BHRRC           — Business & Human Rights Resource Centre (HTML,
                       uses correct /search/ path + real browser UA)

Usage:
    pip install requests beautifulsoup4

    scraper = Scraper("Tesla", 2023, number_of_close_matches=5)
    scraper.scrape()

    run_all()   # all 4 companies × 3 years
"""

import os
import re
import time
import difflib
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import requests
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

COMPANIES = ["Tesla", "Volkswagen", "Disney", "Netflix"]
YEARS     = [2020, 2021, 2022, 2023, 2024, 2025]

OUTPUT_DIR    = Path("ngo_reports")
REQUEST_DELAY = 5.0        # seconds between requests

# Rotate through a few realistic UAs — some sites reject the same string
# repeatedly.  We pick one per Scraper instance.
USER_AGENTS = [
    ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
     "AppleWebKit/537.36 (KHTML, like Gecko) "
     "Chrome/124.0.0.0 Safari/537.36"),
    ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
     "AppleWebKit/605.1.15 (KHTML, like Gecko) "
     "Version/17.4.1 Safari/605.1.15"),
    ("Mozilla/5.0 (X11; Linux x86_64; rv:126.0) "
     "Gecko/20100101 Firefox/126.0"),
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

_ua_index = 0
def _next_ua() -> str:
    global _ua_index
    ua = USER_AGENTS[_ua_index % len(USER_AGENTS)]
    _ua_index += 1
    return ua


# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------

@dataclass
class Report:
    title:       str
    url:         str
    source:      str
    year:        Optional[int] = None
    match_score: float = 0.0
    local_path:  Optional[Path] = None


# ---------------------------------------------------------------------------
# Source adapters
# ---------------------------------------------------------------------------

class WikiRateAdapter:
    """
    WikiRate.org public REST API — returns JSON, no auth needed.
    Endpoint: /companies.json?filter[name]=<company>
    Docs: https://wikirate.org/API_Documentation
    """
    name = "WikiRate"

    def search(self, company: str, year: int) -> list[dict]:
        results = []
        url = "https://wikirate.org/companies.json"
        params = {"filter[name]": company, "limit": 20}
        headers = {"Accept": "application/json", "User-Agent": _next_ua()}
        try:
            r = requests.get(url, params=params, headers=headers, timeout=15)
            r.raise_for_status()
            data = r.json()
            # Each item has 'name' and a canonical URL slug
            for item in (data if isinstance(data, list) else data.get("items", [])):
                name = item.get("name", "")
                slug = item.get("id") or item.get("name", "").replace(" ", "+")
                report_url = f"https://wikirate.org/{slug}"
                results.append({"title": name, "url": report_url})
        except (requests.RequestException, ValueError) as exc:
            log.warning("WikiRate request failed: %s", exc)
        return results


class BHRRCAdapter:
    """
    Business & Human Rights Resource Centre — HTML scrape.

    Bug fixes vs original:
      1. Correct search URL:  /en/search/?q=<company>   (no &type= filter,
         which returned 404 on many queries)
      2. Year filter done post-scrape (BHRRC doesn't support date params
         in the same URL format we used before)
      3. Real browser User-Agent (they return 403 to bots)
      4. Added 'Referer' header to pass their basic bot check
    """
    name = "BHRRC"
    base = "https://www.business-humanrights.org"

    def search(self, company: str, year: int) -> list[dict]:
        results = []
        url = f"{self.base}/en/search/"
        params = {"q": company}
        headers = {
            "User-Agent": _next_ua(),
            "Accept": "text/html,application/xhtml+xml",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": self.base,
        }
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
            r.raise_for_status()
        except requests.RequestException as exc:
            log.warning("BHRRC request failed: %s", exc)
            return results

        soup = BeautifulSoup(r.text, "html.parser")

        # Their results sit in <a> tags inside .search-result or article elements
        # We look broadly then filter by year in the title/snippet
        seen = set()
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/en/search/" in href or href in ("#", "/"):
                continue
            title = a.get_text(strip=True)
            if not title or len(title) < 10:
                continue
            if not href.startswith("http"):
                href = self.base + href
            key = href
            if key not in seen:
                seen.add(key)
                results.append({"title": title, "url": href})

        # Post-filter: keep results whose surrounding text mentions the year
        year_filtered = []
        for item in results:
            if str(year) in item["title"] or str(year) in item["url"]:
                year_filtered.append(item)
        # Fall back to all results if year filter removes everything
        return year_filtered if year_filtered else results


class GoodJobsFirstAdapter:
    """
    Good Jobs First Violation Tracker — https://violationtracker.goodjobsfirst.org
    Public HTML page; lists corporate penalties/violations by company name.
    No year filter available server-side; we return all and let fuzzy
    matching rank them.
    """
    name = "GoodJobsFirst"
    base = "https://violationtracker.goodjobsfirst.org"

    def search(self, company: str, year: int) -> list[dict]:
        results = []
        url = f"{self.base}/prog.php"
        params = {"company_search": company, "order": "pen_year&direction=DESC"}
        headers = {
            "User-Agent": _next_ua(),
            "Accept": "text/html",
            "Referer": self.base,
        }
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
            r.raise_for_status()
        except requests.RequestException as exc:
            log.warning("GoodJobsFirst request failed: %s", exc)
            return results

        soup = BeautifulSoup(r.text, "html.parser")
        for row in soup.select("table tr"):
            cells = row.find_all("td")
            if not cells:
                continue
            # Typical columns: company | agency | penalty | year | description
            # Year is usually in cell index 3
            row_year = None
            for cell in cells:
                text = cell.get_text(strip=True)
                if re.fullmatch(r"\d{4}", text):
                    row_year = int(text)
                    break

            if row_year and row_year != year:
                continue

            a = row.find("a", href=True)
            if a:
                href = a["href"]
                if not href.startswith("http"):
                    href = self.base + "/" + href.lstrip("/")
                title_parts = [c.get_text(strip=True) for c in cells[:3] if c.get_text(strip=True)]
                title = " — ".join(title_parts) if title_parts else a.get_text(strip=True)
                results.append({"title": title, "url": href})

        return results


class OpenSanctionsAdapter:
    """
    OpenSanctions public API — https://www.opensanctions.org/docs/api/
    Free tier: entity search by name, returns JSON.
    Captures sanctions, PEP lists, and watchlist mentions.
    """
    name = "OpenSanctions"
    base = "https://api.opensanctions.org"

    def search(self, company: str, year: int) -> list[dict]:
        results = []
        url = f"{self.base}/match/default"
        # POST body for entity matching
        payload = {
            "queries": {
                "q": {
                    "schema": "Company",
                    "properties": {"name": [company]},
                }
            }
        }
        headers = {
            "User-Agent": _next_ua(),
            "Accept": "application/json",
            "Content-Type": "application/json",
        }
        try:
            r = requests.post(url, json=payload, headers=headers, timeout=20)
            r.raise_for_status()
            data = r.json()
            responses = data.get("responses", {}).get("q", {}).get("results", [])
            for entity in responses:
                name = entity.get("caption", company)
                entity_id = entity.get("id", "")
                report_url = f"https://www.opensanctions.org/entities/{entity_id}/"
                results.append({"title": f"{name} — OpenSanctions profile", "url": report_url})
        except (requests.RequestException, ValueError) as exc:
            log.warning("OpenSanctions request failed: %s", exc)
        return results


ADAPTERS = [
    WikiRateAdapter(),
    BHRRCAdapter(),
    GoodJobsFirstAdapter(),
    OpenSanctionsAdapter(),
]


# ---------------------------------------------------------------------------
# Main Scraper class
# ---------------------------------------------------------------------------

class Scraper:
    """
    Scrapes NGO/accountability sources for reports about `company` in `year`.

    Parameters
    ----------
    company : str
        Company name (e.g. "Tesla").
    year : int
        Target publication year.
    number_of_close_matches : int
        Maximum results to keep after fuzzy scoring.
    output_dir : Path | str
        Root save folder.  Files land in <output_dir>/<company>/<year>/.
    min_score : float
        Minimum fuzzy-match ratio to keep a result (0–1).
    """

    def __init__(
        self,
        company: str,
        year: int,
        number_of_close_matches: int = 5,
        output_dir: Path | str = OUTPUT_DIR,
        min_score: float = 0.3,
    ):
        self.company   = company
        self.year      = year
        self.number_of_close_matches = number_of_close_matches
        self.output_dir = Path(output_dir) / company / str(year)
        self.min_score  = min_score

        self.reports:     list[Report] = []
        self.saved_paths: list[Path]   = []

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def scrape(self) -> list[Report]:
        self.output_dir.mkdir(parents=True, exist_ok=True)
        log.info("Searching '%s' (%d) across %d sources…",
                 self.company, self.year, len(ADAPTERS))

        raw: list[dict] = []
        for adapter in ADAPTERS:
            log.info("  → %s", adapter.name)
            try:
                hits = adapter.search(self.company, self.year)
            except Exception as exc:
                log.warning("  %s raised unexpected error: %s", adapter.name, exc)
                hits = []
            for h in hits:
                h.setdefault("source", adapter.name)
            raw.extend(hits)
            time.sleep(REQUEST_DELAY)

        # Score & filter
        candidates: list[Report] = []
        for item in raw:
            score = self._match_score(item["title"])
            if score >= self.min_score:
                candidates.append(Report(
                    title=item["title"],
                    url=item["url"],
                    source=item.get("source", "unknown"),
                    year=self.year,
                    match_score=round(score, 3),
                ))

        candidates.sort(key=lambda r: r.match_score, reverse=True)
        self.reports = candidates[: self.number_of_close_matches]

        if not self.reports:
            log.warning(
                "No matches for '%s' (%d). Tip: lower min_score (currently %.2f) "
                "or increase number_of_close_matches.",
                self.company, self.year, self.min_score,
            )
        else:
            for report in self.reports:
                self._download(report)
                time.sleep(REQUEST_DELAY)

        n = len(self.saved_paths)
        print(f"\n✓  {n} file(s) for '{self.company}' ({self.year}) → {self.output_dir}")
        return self.reports

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def _match_score(self, title: str) -> float:
        """1.0 if company name is a substring; else SequenceMatcher ratio."""
        if self.company.lower() in title.lower():
            return 1.0
        return difflib.SequenceMatcher(
            None, self.company.lower(), title.lower()
        ).ratio()

    def _download(self, report: Report) -> None:
        """Download report.url → local file.  Skips if already exists."""
        safe = re.sub(r"[^\w\-]+", "_", report.title)[:80]
        headers = {
            "User-Agent": _next_ua(),
            "Accept": "text/html,application/pdf,*/*",
            "Referer": "https://www.google.com/",   # helps with hotlink protection
        }
        try:
            r = requests.get(report.url, headers=headers, timeout=20,
                             stream=True, allow_redirects=True)
            r.raise_for_status()
        except requests.RequestException as exc:
            log.warning("Download failed for '%s': %s", report.url, exc)
            return

        ct = r.headers.get("content-type", "")
        ext = ".pdf" if ("pdf" in ct or report.url.lower().endswith(".pdf")) else ".html"
        dest = self.output_dir / f"{safe}{ext}"

        if dest.exists():
            log.info("    skip (exists): %s", dest.name)
        else:
            with open(dest, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            log.info("    saved: %s", dest.name)

        report.local_path = dest
        self.saved_paths.append(dest)

    def __repr__(self) -> str:
        return (
            f"Scraper(company={self.company!r}, year={self.year}, "
            f"close_matches={self.number_of_close_matches})"
        )


# ---------------------------------------------------------------------------
# Bulk runner
# ---------------------------------------------------------------------------

def run_all(
    companies: list[str] = COMPANIES,
    years:     list[int] = YEARS,
    number_of_close_matches: int = 5,
) -> None:
    total = 0
    for company in companies:
        for year in years:
            s = Scraper(company, year, number_of_close_matches)
            s.scrape()
            total += len(s.saved_paths)
    print(f"\n{'='*55}\nDone — {total} file(s) saved in total.")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    run_all()

22:05:37  INFO      Searching 'Tesla' (2020) across 4 sources…
22:05:37  INFO        → WikiRate
22:05:52  INFO        → BHRRC
22:06:00  WARNING   BHRRC request failed: 403 Client Error: Forbidden for url: https://www.business-humanrights.org/en/search/?q=Tesla
22:06:05  INFO        → GoodJobsFirst
22:06:11  WARNING   GoodJobsFirst request failed: 404 Client Error: Not Found for url: https://violationtracker.goodjobsfirst.org/prog.php?company_search=Tesla&order=pen_year%26direction%3DDESC
22:06:16  INFO        → OpenSanctions
22:06:22  WARNING   OpenSanctions request failed: 401 Client Error: Unauthorized for url: https://api.opensanctions.org/match/default
22:06:33  WARNING   Download failed for 'https://wikirate.org/6273': 404 Client Error: Not Found for url: https://wikirate.org/6273
22:06:44  WARNING   Download failed for 'https://wikirate.org/8220974': 404 Client Error: Not Found for url: https://wikirate.org/8220974
22:06:56  WARNING   Download failed for 'https://wikirate.org/973


✓  0 file(s) for 'Tesla' (2020) → ngo_reports\Tesla\2020


22:07:21  INFO        → BHRRC
22:07:24  WARNING   BHRRC request failed: 403 Client Error: Forbidden for url: https://www.business-humanrights.org/en/search/?q=Tesla
22:07:29  INFO        → GoodJobsFirst
22:07:33  WARNING   GoodJobsFirst request failed: 404 Client Error: Not Found for url: https://violationtracker.goodjobsfirst.org/prog.php?company_search=Tesla&order=pen_year%26direction%3DDESC
22:07:38  INFO        → OpenSanctions
22:07:41  WARNING   OpenSanctions request failed: 401 Client Error: Unauthorized for url: https://api.opensanctions.org/match/default
22:07:49  WARNING   Download failed for 'https://wikirate.org/6273': 404 Client Error: Not Found for url: https://wikirate.org/6273
22:07:58  WARNING   Download failed for 'https://wikirate.org/8220974': 404 Client Error: Not Found for url: https://wikirate.org/8220974
22:08:06  WARNING   Download failed for 'https://wikirate.org/9736206': 404 Client Error: Not Found for url: https://wikirate.org/9736206
22:08:14  WARNING   Dow


✓  0 file(s) for 'Tesla' (2021) → ngo_reports\Tesla\2021


22:08:27  INFO        → BHRRC
22:08:31  WARNING   BHRRC request failed: 403 Client Error: Forbidden for url: https://www.business-humanrights.org/en/search/?q=Tesla
22:08:36  INFO        → GoodJobsFirst
22:08:40  WARNING   GoodJobsFirst request failed: 404 Client Error: Not Found for url: https://violationtracker.goodjobsfirst.org/prog.php?company_search=Tesla&order=pen_year%26direction%3DDESC
22:08:45  INFO        → OpenSanctions
22:08:48  WARNING   OpenSanctions request failed: 401 Client Error: Unauthorized for url: https://api.opensanctions.org/match/default
22:09:03  WARNING   Download failed for 'https://wikirate.org/6273': 404 Client Error: Not Found for url: https://wikirate.org/6273
22:09:15  WARNING   Download failed for 'https://wikirate.org/8220974': 404 Client Error: Not Found for url: https://wikirate.org/8220974
22:09:27  WARNING   Download failed for 'https://wikirate.org/9736206': 404 Client Error: Not Found for url: https://wikirate.org/9736206
22:09:38  WARNING   Dow

...


✓  0 file(s) for 'Volkswagen' (2020) → ngo_reports\Volkswagen\2020


22:14:11  INFO        → BHRRC
22:14:12  WARNING   BHRRC request failed: 403 Client Error: Forbidden for url: https://www.business-humanrights.org/en/search/?q=Volkswagen
22:14:17  INFO        → GoodJobsFirst
22:14:19  WARNING   GoodJobsFirst request failed: 404 Client Error: Not Found for url: https://violationtracker.goodjobsfirst.org/prog.php?company_search=Volkswagen&order=pen_year%26direction%3DDESC
22:14:24  INFO        → OpenSanctions
22:14:26  WARNING   OpenSanctions request failed: 401 Client Error: Unauthorized for url: https://api.opensanctions.org/match/default
22:14:33  WARNING   Download failed for 'https://wikirate.org/47315': 404 Client Error: Not Found for url: https://wikirate.org/47315
22:14:39  WARNING   Download failed for 'https://wikirate.org/2624213': 404 Client Error: Not Found for url: https://wikirate.org/2624213
22:14:46  WARNING   Download failed for 'https://wikirate.org/3152855': 404 Client Error: Not Found for url: https://wikirate.org/3152855
22:14:52  W


✓  0 file(s) for 'Volkswagen' (2021) → ngo_reports\Volkswagen\2021


22:15:11  INFO        → BHRRC
22:15:12  WARNING   BHRRC request failed: 403 Client Error: Forbidden for url: https://www.business-humanrights.org/en/search/?q=Volkswagen
22:15:18  INFO        → GoodJobsFirst
22:15:19  WARNING   GoodJobsFirst request failed: 404 Client Error: Not Found for url: https://violationtracker.goodjobsfirst.org/prog.php?company_search=Volkswagen&order=pen_year%26direction%3DDESC
22:15:24  INFO        → OpenSanctions
22:15:29  WARNING   OpenSanctions request failed: 401 Client Error: Unauthorized for url: https://api.opensanctions.org/match/default


KeyboardInterrupt: 